# Periodogram 

## This notebook aims to perform an analysis of the seasonal component of each time series for each patient.


For this purpose, we define our time series as follows 

$$

z_t = \sum_{t=1}^{T/2} A_j \sin(\omega_j t) + B_j \cos(\omega_j t)

$$



The upper limit of the summation is determined by the definition of the seasonal periods $s_j$. Specifically, we define:
$$
s_j = \frac{T}{j}, \quad \text{for } j = 1, 2, \dots, \left\lfloor \frac{T}{2} \right\rfloor,
$$
which implies that the set of possible seasonal periods is:
$$
s = \{2, 3, \dots, T\}.
$$

From this, we can derive bounds on the corresponding seasonal frequencies $f_j = 1/s_j$. Since $2 \leq s_j \leq T$, it follows that:
$$
\frac{1}{T} \leq f_j \leq \frac{1}{2}.
$$



In this way we can compute all coefficients $$ A_j $$ and $$ B_j $$ for all basic frequencies as the first equation establish. As we're interested at the waves with high amplitudes, we use a function of the frequency to discover those amplitudes, we compute it as follows:

$$
I(f_j) = \frac{T}{2} \, \hat{R}_j^2,
$$

Where:

$$
    \hat{R_j}^2 = \hat{A_j}^2 + \hat{B_j}^2
$$

And :

$$
\hat{A}_j = \frac{2}{T} \sum_{t=1}^{T} z_t \sin(\omega_j t), \qquad
\hat{B}_j = \frac{2}{T} \sum_{t=1}^{T} z_t \cos(\omega_j t),
$$

with $\omega_j = 2\pi f_j = \frac{2\pi j}{T}$ for $j = 1, \dots, \lfloor T/2 \rfloor$.

In [134]:
import pandas as pd 
import matplotlib.pyplot as plt
import numpy as np 

In [135]:
zt = pd.read_csv('../Data/Preprocessed/HUPA0025P.csv', sep=';')['glucose']

In [136]:
def compute_coefficients(zt, frequencies):
    T = len(zt)
    t = np.arange(T)
    
    omega_t = 2 * np.pi * frequencies.reshape(-1, 1) * t
    
    a_j = (2/T) * np.sum(zt.values * np.sin(omega_t), axis=1)
    b_j = (2/T) * np.sum(zt.values * np.cos(omega_t), axis=1)
    
    return a_j, b_j

In [137]:
def periodogram(zt, num_freq):
    T = len(zt)
    frequencies = np.linspace(1/T, 1/2, num=num_freq, dtype=np.float32)
    
    a_j, b_j = compute_coefficients(zt, frequencies)
    periodogram = (T/2) * (np.square(a_j) + np.square(b_j))
    return dict(zip(frequencies, periodogram))

In [138]:
def plot_periodogram(zt):
    T = len(zt)
    num_freq = T//2 + 1
    I_f = periodogram(zt, num_freq)
    
    plt.figure(figsize=(12, 6))
    plt.plot(I_f.keys(), I_f.values())
    plt.xlabel('Frequency')
    plt.ylabel('Amplitude')
    plt.title('Periodogram')
    plt.grid(True)
    plt.show()

In [139]:
T = len(zt)
num_freq = T//2 + 1
periodogram_3 = periodogram(zt, num_freq)

In [140]:
periodogram_3 = dict(sorted(periodogram_3.items(), key=lambda x : x[1], reverse=True))

In [141]:
periodogram_3_season = {1/f:v for f, v in periodogram_3.items()}

In [142]:
periodogram_3_season

{np.float32(400.7801): np.float64(822526.5954069935),
 np.float32(286.27557): np.float64(411779.0239233211),
 np.float32(174.25713): np.float64(389651.35816114605),
 np.float32(222.66054): np.float64(305276.6990587755),
 np.float32(210.94188): np.float64(244867.67664811315),
 np.float32(445.30872): np.float64(196960.6087437943),
 np.float32(2003.5001): np.float64(191198.22515411247),
 np.float32(364.3472): np.float64(178092.27787818192),
 np.float32(1001.8752): np.float64(172073.7497226712),
 np.float32(129.28827): np.float64(161863.50800432122),
 np.float32(308.29593): np.float64(149757.24664299405),
 np.float32(102.76794): np.float64(148361.9696044649),
 np.float32(500.96884): np.float64(145918.45311360512),
 np.float32(182.17772): np.float64(140577.04394188468),
 np.float32(166.99657): np.float64(133351.01605713338),
 np.float32(100.19878): np.float64(111092.26729738445),
 np.float32(138.20454): np.float64(110074.62729877992),
 np.float32(801.52014): np.float64(105139.33502859372),
